# Gram-Type Classification

**- for every column mapping a specific function (e.g., host_type_lysin), it cross-references the matching split history. A score is preserved only if its constituent protein was genuinely held out (split index 0) across all sub-models for that function.**

We build one envelope trained on true-label NC score vectors:

- For a **gram-neg** phage: NC = raw score (high gram-pos score is surprising for gram-neg)
- For a **gram-pos** phage: NC = 1 − raw score (low gram-pos score is surprising for gram-pos)

After this transformation, both gram types cluster near the origin, so we construct a single unified envelope.

### At Testing
For each test phage we create two candidate vectors:
1. Scores as-is → hypothesis: gram-neg
2. 1 − scores   → hypothesis: gram-pos

Both, one, or neither may fall inside the envelope, giving prediction sets of size 2, 1, or 0.

In [1]:
import os
import pickle
import time
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

### Loading the Data

In [2]:
SCORES_PATH = "../../Data/host_type_all_preds.csv"
METADATA_PATH = "../../Data/metadata.csv"
SPLITS_PATH = "../../Data/all_random_pc_splits.pkl"
alpha = 0.1  # Error rate tolerance (Target coverage >= 0.90)

print("Loading scores...")
t0 = time.time()
scores_df = pd.read_csv(SCORES_PATH, index_col="proteinID")
print(f"  scores shape : {scores_df.shape}  ({time.time()-t0:.1f}s)")

print("Loading metadata...")
t0 = time.time()
meta_df = pd.read_csv(METADATA_PATH, index_col="proteinID")
meta_df = meta_df[["accession", "host_type", "split"]]
print(f"  metadata shape : {meta_df.shape}  ({time.time()-t0:.1f}s)")

print("Loading splits...")
t0 = time.time()
with open(SPLITS_PATH, "rb") as f:
    pc_splits = pickle.load(f)

pc_splits = pc_splits.reindex(scores_df.index)
print(f"  splits shape : {pc_splits.shape}  ({time.time()-t0:.1f}s)")

Loading scores...
  scores shape : (1853074, 40)  (3.2s)
Loading metadata...
  metadata shape : (1853074, 3)  (3.4s)
Loading splits...
  splits shape : (1853074, 2160)  (4.5s)


In [3]:
print("Host Type All Preds File (scores)")
scores_df = pd.read_csv(SCORES_PATH, index_col=0)
print(f"  scores shape : {scores_df.shape}  ")
print(scores_df.head(10))

Host Type All Preds File (scores)
  scores shape : (1853074, 40)  
                host_type_lysin  host_type_endolysin  host_type_val  \
proteinID                                                             
AB002632_00001              NaN                  NaN            NaN   
AB002632_00002              NaN                  NaN            NaN   
AB002632_00003              NaN                  NaN            NaN   
AB002632_00004              NaN                  NaN            NaN   
AB002632_00005              NaN                  NaN            NaN   
AB002632_00006              NaN                  NaN            NaN   
AB002632_00007              NaN                  NaN            NaN   
AB002632_00008              NaN                  NaN            NaN   
AB002632_00009              NaN                  NaN            NaN   
AB002632_00010              NaN                  NaN            NaN   

                host_type_capsid  host_type_baseplate  \
proteinID              

In [4]:
print("Metadata...")
meta_df = pd.read_csv(METADATA_PATH, index_col="proteinID")
print(f"  metadata shape : {meta_df.shape}")
print(meta_df.head(10))

Metadata...
  metadata shape : (1853074, 8)
                            pc accession      vc    host host_type  \
proteinID                                                            
AB002632_00001  KP972568_00002  AB002632  VC_0_0  Vibrio  gram-neg   
AB002632_00002  OP297622_00003  AB002632  VC_0_0  Vibrio  gram-neg   
AB002632_00003  KC357596_00003  AB002632  VC_0_0  Vibrio  gram-neg   
AB002632_00004  AB002632_00004  AB002632  VC_0_0  Vibrio  gram-neg   
AB002632_00005  AB002632_00005  AB002632  VC_0_0  Vibrio  gram-neg   
AB002632_00006  AB002632_00006  AB002632  VC_0_0  Vibrio  gram-neg   
AB002632_00007  AB002632_00007  AB002632  VC_0_0  Vibrio  gram-neg   
AB002632_00008    J02451_00006  AB002632  VC_0_0  Vibrio  gram-neg   
AB002632_00009  AB002632_00009  AB002632  VC_0_0  Vibrio  gram-neg   
AB002632_00010  AB002632_00010  AB002632  VC_0_0  Vibrio  gram-neg   

                                   phrogs_annotation  \
proteinID                                              
AB0

In [5]:
print("All Random PC Splits File")
with open(SPLITS_PATH, "rb") as f:
    pc_splits = pickle.load(f)
print(type(pc_splits))
print(pc_splits.shape if hasattr(pc_splits, 'shape') else "not an array")
print(pc_splits.head(10))

All Random PC Splits File
<class 'pandas.DataFrame'>
(1853074, 2160)
                split_Campylobacter_lysin  split_Campylobacter_endolysin  \
proteinID                                                                  
AB002632_00001                       <NA>                           <NA>   
AB002632_00002                       <NA>                           <NA>   
AB002632_00003                       <NA>                           <NA>   
AB002632_00004                       <NA>                           <NA>   
AB002632_00005                       <NA>                           <NA>   
AB002632_00006                       <NA>                           <NA>   
AB002632_00007                       <NA>                           <NA>   
AB002632_00008                       <NA>                           <NA>   
AB002632_00009                       <NA>                           <NA>   
AB002632_00010                       <NA>                           <NA>   

                sp

### Protein-Level Masking

`all_random_pc_splits.pkl` records for every protein and every (host, function) model
which fold that protein was in. Value = 0 means the protein was genuinely held out
(test set) for that model. Any other value means it was in training — its prediction
is contaminated.

For each gram-type score column `host_type_{func}`, we find all splits columns
ending in `_{func}`, and mask the protein's score to NaN if it was in
the training set for ANY of those models. This converts contaminated predictions into
missing data.

In [6]:
score_cols = [c for c in scores_df.columns if c.startswith("host_type_")]

print("Applying protein-level contamination masking...")
t0 = time.time()
scores_masked = scores_df[score_cols].copy()

for col in score_cols:
    func = col.replace("host_type_", "")  # Extracted function, e.g., "lysin"
    matching_split_cols = [c for c in pc_splits.columns if c.endswith("_" + func)]

    if len(matching_split_cols) == 0:
        print(f"  WARNING: no splits columns found for function '{func}' — skipping masking")
        continue

    # A protein is clean to use in the test fold only if it was placed in split-0 across any models generated for that specific function type.
    is_test = (pc_splits[matching_split_cols] == 0).any(axis=1)
    scores_masked[col] = scores_df[col].where(is_test)

print(f"  Done in {time.time()-t0:.1f}s")

original_nans = scores_df[score_cols].isna().sum().sum()
new_nans = scores_masked.isna().sum().sum()
print(f"  NaNs before masking : {original_nans:,}")
print(f"  NaNs after  masking : {new_nans:,}")
print(f"  Newly masked scores : {new_nans - original_nans:,}")

Applying protein-level contamination masking...
  Done in 114.3s
  NaNs before masking : 71,271,481
  NaNs after  masking : 71,324,451
  Newly masked scores : 52,970


In [7]:
# --- DIAGNOSTIC: NaN rate per function group after masking ---
nan_rates = scores_masked[score_cols].isna().mean().sort_values(ascending=False)
print("NaN rate per score column after masking:")
print(nan_rates.to_string())

NaN rate per score column after masking:
host_type_sir2                         0.999713
host_type_lysis_inhibitor              0.997559
host_type_toxin                        0.997399
host_type_replication_initiation       0.996989
host_type_transcriptional_activator    0.996967
host_type_super_infection              0.996553
host_type_annealing                    0.995834
host_type_tail_sheath                  0.995285
host_type_val                          0.994620
host_type_primase                      0.994148
host_type_ejection                     0.993480
host_type_spanin                       0.992901
host_type_portal                       0.990152
host_type_reductase                    0.989823
host_type_endolysin                    0.988474
host_type_DNA_polymerase               0.986011
host_type_holin                        0.984609
host_type_helicase                     0.984606
host_type_lysin                        0.983224
host_type_phosphorylation              0.982004

In [8]:
# --- SUBSET DEFINITIONS FOR DIAGNOSTIC EXPERIMENT ---

# Subset A: biologically informative (likely low NaN rate)
bio_keywords = ["lysin", "tail_appendage", "adsorption"]
subset_bio = [c for c in score_cols if any(kw in c for kw in bio_keywords)]

# Subset B: high-coverage functions (lowest masking, most proteins assigned)
dense_keywords = ["pvp", "dna"] 
subset_dense = [c for c in score_cols if any(kw in c.lower() for kw in dense_keywords)]

# Full set (baseline)
subset_full = score_cols

print("Bio subset columns:", subset_bio)
print("Dense subset columns:", subset_dense)
print(f"\nSizes — full: {len(subset_full)}, bio: {len(subset_bio)}, dense: {len(subset_dense)}")

Bio subset columns: ['host_type_lysin', 'host_type_endolysin', 'host_type_tail_appendage', 'host_type_adsorption-related']
Dense subset columns: ['host_type_DNA_polymerase', 'host_type_pvp', 'host_type_DNA-associated']

Sizes — full: 40, bio: 4, dense: 3


### Protein-to-Phage Aggregation (NaN-aware mean pooling)

In [9]:
combined = scores_masked.join(meta_df, how="inner")

t0 = time.time()
# Mean ignores NaNs, building phage-level scores only from untrained proteins
phage_scores = combined.groupby("accession")[score_cols].mean()
phage_meta = combined.groupby("accession")[["host_type", "split"]].first()
phage_df = phage_scores.join(phage_meta)
phage_df["split"] = phage_df["split"].astype(int)

print(f"Phage-level shape: {phage_df.shape}  ({time.time()-t0:.1f}s)")
print("\nGram-type distribution:")
print(phage_df["host_type"].value_counts())
print("\nSplit distribution:")
print(phage_df["split"].value_counts().sort_index())

Phage-level shape: (18474, 42)  (0.6s)

Gram-type distribution:
host_type
gram-neg    10735
gram-pos     7708
unknown        31
Name: count, dtype: int64

Split distribution:
split
0    4433
1    3173
2    3646
3    3588
4    3634
Name: count, dtype: int64


### Train / Test Split

In [10]:
# train_all_folds_df = phage_df[phage_df["split"] != 0].copy()
# test_df = phage_df[phage_df["split"] == 0].copy()

# train_all_folds_df = train_all_folds_df[train_all_folds_df["host_type"].isin(["gram-neg", "gram-pos"])]
# test_df = test_df[test_df["host_type"].isin(["gram-neg", "gram-pos"])]

# print(f"Training phages (folds 1-4) : {len(train_all_folds_df)}")
# print(f"Test phages     (fold  0)   : {len(test_df)}")

# print(f"\nTrain gram-type distribution:")
# print(train_all_folds_df["host_type"].value_counts())
# print(f"\nTest gram-type distribution:")
# print(test_df["host_type"].value_counts())

In [11]:
# --- 1. Clean Train/Test Split (Fold 0 is Test) ---
train_val_df = phage_df[phage_df["split"] != 0].copy()
test_df_split = phage_df[phage_df["split"] == 0].copy()

# Filter out unknown host types to keep it strictly binary
train_val_df = train_val_df[train_val_df["host_type"].isin(["gram-neg", "gram-pos"])]
test_df_split = test_df_split[test_df_split["host_type"].isin(["gram-neg", "gram-pos"])]

# --- 2. Create the Calibration Split (S1 and S2) ---
# We shuffle the training pool and split it in half
train_shuffled = train_val_df.sample(frac=1, random_state=42)
half_idx = len(train_shuffled) // 2
s1_df = train_shuffled.iloc[:half_idx].copy()
s2_df = train_shuffled.iloc[half_idx:].copy()

print(f"S1 (Envelope Shape Setup) Size : {len(s1_df)}")
print(f"S2 (Threshold Calibration) Size: {len(s2_df)}")
print(f"Test Split (Fold 0) Size       : {len(test_df_split)}")

S1 (Envelope Shape Setup) Size : 7010
S2 (Threshold Calibration) Size: 7011
Test Split (Fold 0) Size       : 4422


In [12]:
# NaN rate per gram type BEFORE imputation, in test set
combined_test = scores_masked.join(meta_df[["accession", "host_type", "split"]], how="inner")
combined_test = combined_test[combined_test["split"] == 0]
phage_nan = combined_test.groupby("accession")[score_cols].apply(lambda x: x.isna().mean().mean())
phage_type = combined_test.groupby("accession")["host_type"].first()

for gt in ["gram-neg", "gram-pos"]:
    mask = phage_type == gt
    print(f"{gt} mean imputed fraction: {phage_nan[mask].mean():.3f}")

gram-neg mean imputed fraction: 0.956
gram-pos mean imputed fraction: 0.961


### NaN Handling and Median Imputation


In [13]:
# # Imputing missing scores with class specific medians for the training set and global medians for the test set (capped of at 0.5 to avoid overconfident imputations)
# valid_cols = score_cols
# K = len(valid_cols)

# # Computing global medians for each score column across the entire training set (ignoring NaNs)
# global_train_medians = train_all_folds_df[valid_cols].median()
# global_safe_impute = global_train_medians.copy()
# global_safe_impute[global_safe_impute > 0.6] = 0.5

# # Computing class-specific medians and perform safe imputation for the training set
# for gt in ["gram-neg", "gram-pos"]:
#     mask = train_all_folds_df["host_type"] == gt
#     class_medians = train_all_folds_df.loc[mask, valid_cols].median()
#     class_safe_impute = class_medians.copy()
#     # class_safe_impute[class_safe_impute > 0.6] = 0.5
#     train_all_folds_df.loc[mask, valid_cols] = train_all_folds_df.loc[mask, valid_cols].fillna(class_safe_impute)

# # 
# test_df[valid_cols] = test_df[valid_cols].fillna(global_safe_impute)
# train_all_folds_df[valid_cols] = train_all_folds_df[valid_cols].fillna(0.5)
# test_df[valid_cols] = test_df[valid_cols].fillna(0.5)

# test_raw = test_df[valid_cols].values
# test_labels = test_df["host_type"].values

In [14]:
# --- 1. Clean Train/Test Split (Fold 0 is Test) ---
train_val_df = phage_df[phage_df["split"] != 0].copy()
test_df_split = phage_df[phage_df["split"] == 0].copy()

# Filter out unknown host types to keep it strictly binary
train_val_df = train_val_df[train_val_df["host_type"].isin(["gram-neg", "gram-pos"])]
test_df_split = test_df_split[test_df_split["host_type"].isin(["gram-neg", "gram-pos"])]

# --- 2. Create the Calibration Split (S1 and S2) ---
train_shuffled = train_val_df.sample(frac=1, random_state=42)
half_idx = len(train_shuffled) // 2
s1_df = train_shuffled.iloc[:half_idx].copy()
s2_df = train_shuffled.iloc[half_idx:].copy()

print(f"S1 (Envelope Shape Setup) Size : {len(s1_df)}")
print(f"S2 (Threshold Calibration) Size: {len(s2_df)}")
print(f"Test Split (Fold 0) Size       : {len(test_df_split)}")

S1 (Envelope Shape Setup) Size : 7010
S2 (Threshold Calibration) Size: 7011
Test Split (Fold 0) Size       : 4422


### Score Characteristics Check

In [15]:
# print("Mean score per gram type (averaged across all K columns):")
# for gt in ["gram-neg", "gram-pos"]:
#     mask = train_all_folds_df["host_type"] == gt
#     mean_score   = train_all_folds_df.loc[mask, valid_cols].mean().mean()
#     median_score = train_all_folds_df.loc[mask, valid_cols].median().median()
#     print(f"  {gt}: mean={mean_score:.4f}, median={median_score:.4f}, n={mask.sum()}")

# col_means = pd.DataFrame({
#     "gram-neg": train_all_folds_df[train_all_folds_df["host_type"]=="gram-neg"][valid_cols].mean(),
#     "gram-pos": train_all_folds_df[train_all_folds_df["host_type"]=="gram-pos"][valid_cols].mean(),
# })
# col_means["gram-neg > gram-pos"] = col_means["gram-neg"] > col_means["gram-pos"]
# print(f"\nColumns where gram-neg scores higher: {col_means['gram-neg > gram-pos'].sum()} / {len(valid_cols)}")

# test_raw    = test_df[valid_cols].values
# test_labels = test_df["host_type"].values

### Envelope Methods

In [16]:
def split_data_by_fold(df, valid_cols, test_fold_id):
    df = df.copy()
    df["split"] = df["split"].astype(int)
    s1_df = df[df["split"] != test_fold_id]
    s2_df = df[df["split"] == test_fold_id]
    return (s1_df[valid_cols].values, s2_df[valid_cols].values, 
            s1_df["host_type"].to_numpy(dtype=str), s2_df["host_type"].to_numpy(dtype=str))

In [17]:
def quantile_for_class(tau_scores, labels, cls, alpha):
    vals = np.sort(tau_scores[labels == cls])
    n = len(vals)
    if n == 0: return 0.0
    idx = int(np.ceil((n + 1) * (1 - alpha))) - 1
    return vals[np.clip(idx, 0, n - 1)]

#### Radial Envelope

In [18]:
def sample_positive_sphere(M, K):
    V = np.abs(np.random.randn(M, K))
    return V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)

def radial_build_envelope(S1, S2, labels_S1, labels_S2, alpha, M=200, delta_deg=30):
    K = S1.shape[1]
    U = sample_positive_sphere(M, K)
    mags = np.linalg.norm(S1, axis=1)
    dirs = S1 / (mags[:, None] + 1e-12)
    cos_thresh = np.cos(np.radians(delta_deg))

    q_tilde = np.array([
        np.quantile(mags[(dirs @ U[m]) >= cos_thresh], 1 - alpha)
        if ((dirs @ U[m]) >= cos_thresh).sum() > 5 else np.quantile(mags, 1 - alpha)
        for m in range(M)
    ])

    tau_scores = ((S2[:, None, :] / (U * q_tilde[:, None])[None, :, :]).max(axis=2).min(axis=1))
    t_hat_neg = quantile_for_class(tau_scores, labels_S2, "gram-neg", alpha)
    t_hat_pos = quantile_for_class(tau_scores, labels_S2, "gram-pos", alpha)
    return {"U": U, "q_tilde": q_tilde, "t_hat": max(t_hat_neg, t_hat_pos), "method": "radial"}

def radial_is_in_region(scores, envelope):
    U, q_tilde, t_hat = envelope["U"], envelope["q_tilde"], envelope["t_hat"]
    boundary = U * (q_tilde * t_hat)[:, None]
    return np.any(np.all(scores[:, None, :] <= boundary[None, :, :], axis=2), axis=1)


#### Strip Envelope

In [19]:
def strip_shape_discovery(S1, alpha, M):
    N1, K = S1.shape
    bin_edges = np.array([np.linspace(S1[:, j].min(), S1[:, j].max(), M + 1) for j in range(K)])
    limits = np.zeros((K, K, M))
    for j in range(K):
        for m in range(M):
            lo, hi = bin_edges[j, m], bin_edges[j, m + 1]
            in_strip = (S1[:, j] >= lo) & (S1[:, j] < hi)
            for i in range(K):
                if i == j: continue
                src = S1[in_strip, i] if in_strip.sum() >= 3 else S1[:, i]
                limits[j, i, m] = np.quantile(src, 1 - alpha)
    for j in range(K):
        for i in range(K):
            if i == j: continue
            for m in range(M - 2, -1, -1):
                limits[j, i, m] = max(limits[j, i, m], limits[j, i, m + 1])
    return bin_edges, limits

def get_bin_indices(scores, bin_edges):
    K = scores.shape[1]
    M = bin_edges.shape[1] - 1
    return np.array([np.clip(np.digitize(scores[:, j], bin_edges[j]) - 1, 0, M - 1) for j in range(K)]).T

def strip_size_scaling(S2, bin_edges, limits):
    N2, K = S2.shape
    bin_idx = get_bin_indices(S2, bin_edges)
    ratios = np.zeros((N2, K, K))
    for j in range(K):
        for i in range(K):
            if i == j: continue
            lims = limits[j, i, bin_idx[:, j]]
            ratios[:, j, i] = S2[:, i] / (lims + 1e-12)
    return ratios.max(axis=(1, 2))

def strip_build_envelope(S1, S2, labels_S1, labels_S2, alpha, M=20):
    bin_edges, limits = strip_shape_discovery(S1, alpha, M)
    tau_scores = strip_size_scaling(S2, bin_edges, limits)
    t_hat_neg = quantile_for_class(tau_scores, labels_S2, "gram-neg", alpha)
    t_hat_pos = quantile_for_class(tau_scores, labels_S2, "gram-pos", alpha)
    return {"bin_edges": bin_edges, "limits": limits, "t_hat": max(t_hat_neg, t_hat_pos), "method": "strip"}


def strip_is_in_region(scores, envelope):
    bin_edges, limits, t_hat = envelope["bin_edges"], envelope["limits"], envelope["t_hat"]
    N, K = scores.shape
    bin_idx = get_bin_indices(scores, bin_edges)
    ratios = np.zeros((N, K, K))
    for j in range(K):
        for i in range(K):
            if i == j: continue
            lims = limits[j, i, bin_idx[:, j]]
            ratios[:, j, i] = scores[:, i] / (lims + 1e-12)
    return ratios.max(axis=(1, 2)) <= t_hat


#### Collapsed 1D Envelope

In [20]:
def collapsed_build_envelope(S1, S2, labels_S1, labels_S2, alpha):
    S1_1d = S1.mean(axis=1, keepdims=True)
    S2_1d = S2.mean(axis=1, keepdims=True)
    q_tilde = np.quantile(S1_1d, 1 - alpha)
    tau_scores = (S2_1d / (q_tilde + 1e-12)).ravel()
    t_hat_neg = quantile_for_class(tau_scores, labels_S2, "gram-neg", alpha)
    t_hat_pos = quantile_for_class(tau_scores, labels_S2, "gram-pos", alpha)
    return {"q_tilde": q_tilde, "t_hat": max(t_hat_neg, t_hat_pos), "method": "collapsed"}


def collapsed_is_in_region(scores, envelope):
    scores_1d = scores.mean(axis=1, keepdims=True)
    return scores_1d.ravel() <= envelope["q_tilde"] * envelope["t_hat"]

### Prediction

In [21]:
def is_in_region(scores, envelope):
    if envelope["method"] == "radial": return radial_is_in_region(scores, envelope)
    elif envelope["method"] == "strip": return strip_is_in_region(scores, envelope)
    return collapsed_is_in_region(scores, envelope)

In [22]:
def predict_gram_type_nc(test_raw, envelope):
    # Negatives evaluate as-is, Positives evaluate flipped (1 - score)
    nc_gramneg = test_raw
    nc_grampos = 1.0 - test_raw

    in_gramneg = is_in_region(nc_gramneg, envelope)
    in_grampos = is_in_region(nc_grampos, envelope)

    results = []
    for i in range(len(test_raw)):
        pred_set = []
        if in_gramneg[i]: pred_set.append("gram-neg")
        if in_grampos[i]: pred_set.append("gram-pos")
        results.append({"in_gramneg": bool(in_gramneg[i]), "in_grampos": bool(in_grampos[i]), "prediction_set": pred_set})
    return results

### Evaluation

In [23]:
def evaluate(results, true_labels, alpha):
    n = len(results)
    set_sizes = [len(r["prediction_set"]) for r in results]
    covered = sum(true_labels[i] in r["prediction_set"] for i, r in enumerate(results))
    singletons = [(r, true_labels[i]) for i, r in enumerate(results) if len(r["prediction_set"]) == 1]
    correct_singletons = sum(r["prediction_set"][0] == lbl for r, lbl in singletons)
    empty = sum(s == 0 for s in set_sizes)
    size2 = sum(s == 2 for s in set_sizes)

    return {
        "coverage": covered / n, "avg_set_size": np.mean(set_sizes),
        "empty_rate": empty / n, "singleton_rate": len(singletons) / n, "size2_rate": size2 / n,
        "singleton_accuracy": (correct_singletons / len(singletons) if singletons else None),
        "set_sizes": set_sizes,
        "per_type_coverage": {
            gt: sum(true_labels[i] in results[i]["prediction_set"] for i in range(n) if true_labels[i] == gt)
            / max(1, sum(lbl == gt for lbl in true_labels)) for gt in ["gram-neg", "gram-pos"]
        },
    }

### Evaluating on split 0

In [24]:
# def run_pipeline_on_subset(phage_df, cols, alpha, label=""):
#     """Run full conformal pipeline on a given subset of score columns."""
    
#     # --- train/test split ---
#     train_df = phage_df[phage_df["split"] != 0].copy()
#     test_df_sub = phage_df[phage_df["split"] == 0].copy()
#     train_df = train_df[train_df["host_type"].isin(["gram-neg", "gram-pos"])]
#     test_df_sub = test_df_sub[test_df_sub["host_type"].isin(["gram-neg", "gram-pos"])]

#     # --- imputation (same logic as before) ---
#     global_train_medians = train_df[cols].median()
#     global_safe_impute = global_train_medians.clip(upper=0.6).where(global_train_medians <= 0.6, 0.5)

#     for gt in ["gram-neg", "gram-pos"]:
#         mask = train_df["host_type"] == gt
#         class_medians = train_df.loc[mask, cols].median()
#         class_safe_impute = class_medians.copy()
#         class_safe_impute[class_safe_impute > 0.6] = 0.5
#         train_df.loc[mask, cols] = train_df.loc[mask, cols].fillna(class_safe_impute)
#     train_df[cols] = train_df[cols].fillna(0.5)
#     test_df_sub[cols] = test_df_sub[cols].fillna(global_safe_impute).fillna(0.5)

#     # --- NaN rate diagnostic for this subset ---
#     nan_frac = test_df_sub[cols].isna().mean().mean()  # should be 0 after imputation
#     imputed_frac = (phage_df[phage_df["split"] == 0][cols].isna().mean(axis=1)).mean()
#     print(f"\n[{label}] columns={len(cols)}, "
#           f"mean imputed fraction per test phage = {imputed_frac:.3f}")

#     # --- calibration split ---
#     train_shuffled = train_df.sample(frac=1)
#     half = len(train_shuffled) // 2
#     final_s1 = train_shuffled.iloc[:half]
#     final_s2 = train_shuffled.iloc[half:]

#     s1_raw = final_s1[cols].values
#     s2_raw = final_s2[cols].values
#     labels_s1 = final_s1["host_type"].to_numpy(dtype=str)
#     labels_s2 = final_s2["host_type"].to_numpy(dtype=str)

#     S1_nc = np.where(labels_s1[:, None] == "gram-neg", s1_raw, 1.0 - s1_raw)
#     S2_nc = np.where(labels_s2[:, None] == "gram-neg", s2_raw, 1.0 - s2_raw)

#     # --- fit envelopes ---
#     r_env = radial_build_envelope(S1_nc, S2_nc, labels_s1, labels_s2, alpha, M=200)
#     s_env = strip_build_envelope(S1_nc, S2_nc, labels_s1, labels_s2, alpha, M=20)
#     c_env = collapsed_build_envelope(S1_nc, S2_nc, labels_s1, labels_s2, alpha)

#     test_raw_sub = test_df_sub[cols].values
#     test_labels_sub = test_df_sub["host_type"].values

#     # --- evaluate ---
#     rows = []
#     for name, key, res in [
#         ("Radial", "radial", predict_gram_type_nc(test_raw_sub, r_env)),
#         ("Strip", "strip", predict_gram_type_nc(test_raw_sub, s_env)),
#         ("Collapsed", "collapsed", predict_gram_type_nc(test_raw_sub, c_env)),
#     ]:
#         m = evaluate(res, test_labels_sub, alpha)
#         rows.append({
#             "Method": name,
#             "Coverage": round(m["coverage"], 3),
#             "Neg cov": round(m["per_type_coverage"]["gram-neg"], 3),
#             "Pos cov": round(m["per_type_coverage"]["gram-pos"], 3),
#             "Avg set size": round(m["avg_set_size"], 3),
#             "Empty %": round(m["empty_rate"] * 100, 1),
#         })
    
#     df_out = pd.DataFrame(rows)
#     print(f"\n=== {label} ===")
#     print(df_out.to_string(index=False))
#     return df_out

In [25]:
print("\n" + "="*55)
print(" LEAKAGE-FREE CONFORMAL CALIBRATION & TESTING ")
print("="*55)

valid_cols = score_cols

# 1. Compute global medians using ONLY S1 (completely blind to S2 and Test)
global_s1_medians = s1_df[valid_cols].median()
global_safe_impute = global_s1_medians.copy()
global_safe_impute[global_safe_impute > 0.6] = 0.5

# 2. Impute S1 using class-specific medians (This builds our clean baseline envelope shape)
for gt in ["gram-neg", "gram-pos"]:
    mask = s1_df["host_type"] == gt
    class_medians = s1_df.loc[mask, valid_cols].median()
    class_safe_impute = class_medians.copy()
    class_safe_impute[class_safe_impute > 0.6] = 0.5
    s1_df.loc[mask, valid_cols] = s1_df.loc[mask, valid_cols].fillna(class_safe_impute)

s1_df[valid_cols] = s1_df[valid_cols].fillna(0.5)

# 3. CRITICAL FIX: Impute S2 and Test_Split using the EXACT SAME global rules from S1
s2_df[valid_cols] = s2_df[valid_cols].fillna(global_safe_impute).fillna(0.5)
test_df_split[valid_cols] = test_df_split[valid_cols].fillna(global_safe_impute).fillna(0.5)

# Extract underlying arrays
s1_raw = s1_df[valid_cols].values
s2_raw = s2_df[valid_cols].values
test_raw = test_df_split[valid_cols].values

labels_s1 = s1_df["host_type"].to_numpy(dtype=str)
labels_s2 = s2_df["host_type"].to_numpy(dtype=str)
test_labels = test_df_split["host_type"].to_numpy(dtype=str)

# 4. Transform to Non-Conformity (NC) Scores
S1_nc = np.where(labels_s1[:, None] == "gram-neg", s1_raw, 1.0 - s1_raw)
S2_nc = np.where(labels_s2[:, None] == "gram-neg", s2_raw, 1.0 - s2_raw)

# 5. Fit Envelopes (Using Collapsed 1D as an example)
final_c_env = collapsed_build_envelope(S1_nc, S2_nc, labels_s1, labels_s2, alpha)

# 6. Predict on the true final test set (Fold 0)
final_c_res = predict_gram_type_nc(test_raw, final_c_env)

# 7. Evaluate Performance
m = evaluate(final_c_res, test_labels, alpha)

print("\n=== DEFINITIVE HELD-OUT TEST RESULTS (FOLD 0) ===")
print(f"Method: Collapsed 1D")
print(f"Overall Coverage: {m['coverage']:.3f}  (Target: {1-alpha:.2f})")
print(f"Gram-Neg Coverage: {m['per_type_coverage']['gram-neg']:.3f}")
print(f"Gram-Pos Coverage: {m['per_type_coverage']['gram-pos']:.3f}")
print(f"Avg Set Size: {m['avg_set_size']:.3f}")
print(f"Empty Set Rate: {m['empty_rate']*100:.1f}%")


 LEAKAGE-FREE CONFORMAL CALIBRATION & TESTING 

=== DEFINITIVE HELD-OUT TEST RESULTS (FOLD 0) ===
Method: Collapsed 1D
Overall Coverage: 0.975  (Target: 0.90)
Gram-Neg Coverage: 0.999
Gram-Pos Coverage: 0.932
Avg Set Size: 0.995
Empty Set Rate: 0.5%


In [26]:
# # Re-use phage_df BEFORE imputation — rebuild it from scores_masked
# # (make sure phage_df still has the original NaN structure here, not post-imputation)

# results_full  = run_pipeline_on_subset(phage_df, subset_full,  alpha, label="FULL")
# results_bio   = run_pipeline_on_subset(phage_df, subset_bio,   alpha, label="BIO (lysin+tail)")
# results_dense = run_pipeline_on_subset(phage_df, subset_dense, alpha, label="DENSE (pvp+DNA)")

In [27]:
print("\n" + "="*55)
print(" LEAKAGE-FREE CONFORMAL CALIBRATION & TESTING ")
print("="*55)

valid_cols = score_cols

# 1. Compute global medians using ONLY S1 (completely blind to S2 and Test)
global_s1_medians = s1_df[valid_cols].median()
global_safe_impute = global_s1_medians.copy()
global_safe_impute[global_safe_impute > 0.6] = 0.5

# 2. Impute S1 using class-specific medians (This builds our clean baseline envelope shape)
for gt in ["gram-neg", "gram-pos"]:
    mask = s1_df["host_type"] == gt
    class_medians = s1_df.loc[mask, valid_cols].median()
    class_safe_impute = class_medians.copy()
    class_safe_impute[class_safe_impute > 0.6] = 0.5
    s1_df.loc[mask, valid_cols] = s1_df.loc[mask, valid_cols].fillna(class_safe_impute)

s1_df[valid_cols] = s1_df[valid_cols].fillna(0.5)

# 3. CRITICAL FIX: Impute S2 and Test_Split using the EXACT SAME global rules from S1
s2_df[valid_cols] = s2_df[valid_cols].fillna(global_safe_impute).fillna(0.5)
test_df_split[valid_cols] = test_df_split[valid_cols].fillna(global_safe_impute).fillna(0.5)

# Extract underlying arrays
s1_raw = s1_df[valid_cols].values
s2_raw = s2_df[valid_cols].values
test_raw = test_df_split[valid_cols].values

labels_s1 = s1_df["host_type"].to_numpy(dtype=str)
labels_s2 = s2_df["host_type"].to_numpy(dtype=str)
test_labels = test_df_split["host_type"].to_numpy(dtype=str)

# 4. Transform to Non-Conformity (NC) Scores
S1_nc = np.where(labels_s1[:, None] == "gram-neg", s1_raw, 1.0 - s1_raw)
S2_nc = np.where(labels_s2[:, None] == "gram-neg", s2_raw, 1.0 - s2_raw)

# 5. Fit Envelopes (Using Collapsed 1D as our target example)
final_c_env = collapsed_build_envelope(S1_nc, S2_nc, labels_s1, labels_s2, alpha)

# 6. Predict on the true final test set (Fold 0)
final_c_res = predict_gram_type_nc(test_raw, final_c_env)

# 7. Evaluate Performance
m = evaluate(final_c_res, test_labels, alpha)

print("\n=== DEFINITIVE HELD-OUT TEST RESULTS (FOLD 0) ===")
print(f"Method: Collapsed 1D")
print(f"Overall Coverage: {m['coverage']:.3f}  (Target: {1-alpha:.2f})")
print(f"Gram-Neg Coverage: {m['per_type_coverage']['gram-neg']:.3f}")
print(f"Gram-Pos Coverage: {m['per_type_coverage']['gram-pos']:.3f}")
print(f"Avg Set Size: {m['avg_set_size']:.3f}")
print(f"Empty Set Rate: {m['empty_rate']*100:.1f}%")


 LEAKAGE-FREE CONFORMAL CALIBRATION & TESTING 

=== DEFINITIVE HELD-OUT TEST RESULTS (FOLD 0) ===
Method: Collapsed 1D
Overall Coverage: 0.975  (Target: 0.90)
Gram-Neg Coverage: 0.999
Gram-Pos Coverage: 0.932
Avg Set Size: 0.995
Empty Set Rate: 0.5%
